In [ ]:
# ── Imports and setup ──

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.stats import hypergeom
from scipy.stats import f_oneway, kruskal
from scipy.stats import mannwhitneyu
from itertools import combinations
from scipy.stats import false_discovery_control
import os

BASE_DIR = os.path.dirname(os.path.abspath("__file__"))
OUT = os.path.join(BASE_DIR, "outputs")
os.makedirs(OUT, exist_ok=True)

# Input files
TEMPTED_LOADINGS   = os.path.join(OUT, "tempted_subject_loadings.csv")
METADATA           = os.path.join(OUT, "merged_metadata_clean.csv")
SHAP_RANKED        = os.path.join(OUT, "delta_feature_importances.csv")
LME_SIG            = os.path.join(OUT, "lme_significant_otus_annotated.csv")
LME_ALL            = os.path.join(OUT, "lme_interaction_results.csv")
TAXONOMY           = os.path.join(OUT, "taxonomy_table.csv")

# Output files (this notebook)
OUT_TEMPTED_PLOT   = os.path.join(OUT, "tempted_fibertype_plot.png")
OUT_OVERLAP_PLOT   = os.path.join(OUT, "lme_shap_overlap.png")
OUT_OVERLAP_TABLE  = os.path.join(OUT, "lme_shap_overlap_table.csv")

print("Paths set.")
print(f"  Base : {BASE_DIR}")
print(f"  Out  : {OUT}")

In [ ]:
# ── Load data ──

# TEMPTED subject loadings (529 subjects × PC1, PC2, treatment, study)
tempted = pd.read_csv(TEMPTED_LOADINGS)
print(f"TEMPTED loadings : {tempted.shape}  cols: {list(tempted.columns)}")

# Merged metadata — for joining fiber_type to subjects
meta = pd.read_csv(METADATA, low_memory=False)
print(f"Metadata         : {meta.shape}  cols: {list(meta.columns[:10])}")

# SHAP rankings — 9,612 OTUs ranked by mean |SHAP| from delta RF
shap_df = pd.read_csv(SHAP_RANKED)
print(f"SHAP rankings    : {shap_df.shape}")

# LME significant OTUs (96 rows, FDR < 0.05, with taxonomy)
lme_sig = pd.read_csv(LME_SIG)
print(f"LME significant  : {lme_sig.shape}")

# Full LME results (200 OTUs, for background set)
lme_all = pd.read_csv(LME_ALL)
print(f"LME all OTUs     : {lme_all.shape}")

# Taxonomy table
tax = pd.read_csv(TAXONOMY, low_memory=False)
if tax.columns[0] != 'OTU_ID':
    tax = tax.rename(columns={tax.columns[0]: 'OTU_ID'})
print(f"Taxonomy         : {tax.shape}")

In [ ]:
# ── Fiber type group mapping ──

FIBER_GROUP_MAP = {
    # Group 1 — Prebiotic oligosaccharides
    'inulin'                              : 1,
    'long_chain_inulin'                   : 1,
    'short_chain_FOS'                     : 1,
    'FOS'                                 : 1,
    'GOS'                                 : 1,
    'oligofructose'                       : 1,
    'psyllium'                            : 1,
    # Group 2 — Resistant starch
    'himaize'                             : 2,
    'potato'                              : 2,
    'potato_RS4A'                         : 2,
    'potato_RS4B'                         : 2,
    'potato_RS4C'                         : 2,
    'potato_starch'                       : 2,
    'maize'                               : 2,
    'tapioca'                             : 2,
    'starch-entrapped-microspheres-12g'   : 2,
    'starch-entrapped-microspheres-9g'    : 2,
    # Group 3 — Soluble fiber
    'soluble_corn'                        : 3,
    'polydextrose'                        : 3,
    # Group 4 — Grain / wheat fiber
    'barley-kernel-bread'                 : 4,
    'white-wheat-bread'                   : 4,
    # Group 5 — Mixed / unspecified
    'fiber_diet_10g'                      : 5,
    'fiber_diet_40g'                      : 5,
}

FIBER_GROUP_LABELS = {
    1 : 'Prebiotic oligosaccharides',
    2 : 'Resistant starch',
    3 : 'Soluble fiber',
    4 : 'Grain / wheat fiber',
    5 : 'Mixed / unspecified',
}

GROUP_COLORS = {
    1 : '#E07B54',
    2 : '#5B8DB8',
    3 : '#6DBF67',
    4 : '#B07CC6',
    5 : '#F0C04A',
    0 : '#AAAAAA',   # control / unmapped
}

# Verify — any fiber types still unmapped?
all_fiber = [f for f in meta['fiber_type'].dropna().unique()
             if f not in ['corn_control','maltodextrin_control','starch_control','none']]
unmapped = [f for f in all_fiber if f not in FIBER_GROUP_MAP]
print(f"Unmapped fiber types: {unmapped if unmapped else 'None — all mapped'}")
print(f"Total mapped entries: {len(FIBER_GROUP_MAP)}")

In [ ]:
# ── Join fiber type to TEMPTED loadings ──

# Get one metadata row per subject (after-timepoint preferred)
meta_sub = (
    meta[meta['timepoint'] == 'after']
    [['subject_id', 'fiber_type', 'study', 'treatment']]
    .drop_duplicates(subset='subject_id')
)

print(f"Metadata subjects (after-timepoint): {len(meta_sub)}")
print(f"TEMPTED subjects                    : {len(tempted)}")

# Standardise subject_id for merge
tempted['subject_id_lower'] = tempted['subject_id'].str.lower().str.strip()
meta_sub = meta_sub.copy()
meta_sub['subject_id_lower'] = meta_sub['subject_id'].str.lower().str.strip()

# Merge
tempted_ann = tempted.merge(
    meta_sub[['subject_id_lower', 'fiber_type']],
    on='subject_id_lower', how='left'
)

# Map to group number
tempted_ann['fiber_group'] = (
    tempted_ann['fiber_type']
    .map(FIBER_GROUP_MAP)
    .fillna(0)
    .astype(int)
)

n_matched   = tempted_ann['fiber_type'].notna().sum()
n_unmatched = tempted_ann['fiber_type'].isna().sum()
print(f"\nFiber type join: {n_matched} matched, {n_unmatched} unmatched")
print("\nSubjects per group:")
print(tempted_ann['fiber_group'].value_counts().sort_index())

In [ ]:
# ── Figure: TEMPTED subject loadings (Fig. 3) ──

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('TEMPTED Subject Loadings', fontsize=14, fontweight='bold', y=1.01)

# ── Panel A ───────────────────────────────────────────────
ax = axes[0]
colors_ab = {'fiber': '#E07B54', 'control': '#5B8DB8'}
for arm, grp in tempted_ann.groupby('treatment'):
    ax.scatter(grp['PC1'], grp['PC2'],
               c=colors_ab.get(arm, '#999999'),
               alpha=0.6, s=30, label=arm.capitalize(), edgecolors='none')
ax.set_xlabel('Component 1 (PC1)', fontsize=11)
ax.set_ylabel('Component 2 (PC2)', fontsize=11)
ax.set_title('A   Fiber vs Control', fontsize=12, loc='left')
ax.legend(frameon=False, fontsize=10,
          bbox_to_anchor=(0.5, -0.15), loc='upper center', ncol=2)
ax.spines[['top', 'right']].set_visible(False)

# ── Panel B ───────────────────────────────────────────────
ax = axes[1]

ctrl = tempted_ann[tempted_ann['treatment'] == 'control']
ax.scatter(ctrl['PC1'], ctrl['PC2'],
           c='#CCCCCC', alpha=0.35, s=25, label='Control',
           edgecolors='none', zorder=1)

fiber = tempted_ann[tempted_ann['treatment'] == 'fiber'].copy()
for grp_id in sorted(fiber['fiber_group'].unique()):
    sub = fiber[fiber['fiber_group'] == grp_id]
    label = FIBER_GROUP_LABELS.get(grp_id, 'Unmapped') if grp_id != 0 else 'Unmapped'
    ax.scatter(sub['PC1'], sub['PC2'],
               c=GROUP_COLORS.get(grp_id, '#999999'),
               alpha=0.65, s=35, label=f'{label} (n={len(sub)})',
               edgecolors='none', zorder=2)

ax.set_xlabel('Component 1 (PC1)', fontsize=11)
ax.set_ylabel('Component 2 (PC2)', fontsize=11)
ax.set_title('B   Fiber Type Groups', fontsize=12, loc='left')
ax.legend(frameon=False, fontsize=9,
          bbox_to_anchor=(0.5, -0.15), loc='upper center', ncol=3)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig(OUT_TEMPTED_PLOT, dpi=200, bbox_inches='tight')
plt.show()
print(f"Saved: {OUT_TEMPTED_PLOT}")

In [ ]:
# ── Group separation test (ANOVA + Kruskal–Wallis) ──

# Fiber arm only, exclude unmapped (group 0)
fiber_only = tempted_ann[
    (tempted_ann['treatment'] == 'fiber') &
    (tempted_ann['fiber_group'] != 0)
].reset_index(drop=True)

print(f"Subjects entering test : {len(fiber_only)}")
print(f"Group sizes:")
print(fiber_only['fiber_group'].value_counts().sort_index())

# Extract PC1 and PC2 values per group
groups_pc1 = [g['PC1'].values for _, g in fiber_only.groupby('fiber_group')]
groups_pc2 = [g['PC2'].values for _, g in fiber_only.groupby('fiber_group')]

# One-way ANOVA (parametric)
F_pc1, p_pc1 = f_oneway(*groups_pc1)
F_pc2, p_pc2 = f_oneway(*groups_pc2)

# Kruskal-Wallis (non-parametric — more appropriate for non-normal data)
H_pc1, kp_pc1 = kruskal(*groups_pc1)
H_pc2, kp_pc2 = kruskal(*groups_pc2)

print(f"\nOne-way ANOVA:")
print(f"  PC1 : F = {F_pc1:.4f},  p = {p_pc1:.4e}")
print(f"  PC2 : F = {F_pc2:.4f},  p = {p_pc2:.4e}")

print(f"\nKruskal-Wallis (non-parametric):")
print(f"  PC1 : H = {H_pc1:.4f},  p = {kp_pc1:.4e}")
print(f"  PC2 : H = {kp_pc2:.4f},  p = {kp_pc2:.4e}")

In [ ]:
# ── Post-hoc pairwise Mann–Whitney U (PC2) ──

groups     = sorted(fiber_only['fiber_group'].unique())
pairs      = list(combinations(groups, 2))
pair_results = []

for g1, g2 in pairs:
    pc2_g1 = fiber_only[fiber_only['fiber_group'] == g1]['PC2'].values
    pc2_g2 = fiber_only[fiber_only['fiber_group'] == g2]['PC2'].values
    U, p   = mannwhitneyu(pc2_g1, pc2_g2, alternative='two-sided')
    pair_results.append({
        'group_1'  : f"{g1} — {FIBER_GROUP_LABELS[g1]}",
        'group_2'  : f"{g2} — {FIBER_GROUP_LABELS[g2]}",
        'n_g1'     : len(pc2_g1),
        'n_g2'     : len(pc2_g2),
        'U'        : round(U, 2),
        'p_raw'    : p,
    })

posthoc = pd.DataFrame(pair_results)

# BH FDR correction
posthoc['q_value'] = false_discovery_control(posthoc['p_raw'], method='bh')
posthoc['significant'] = posthoc['q_value'] < 0.05

print("Post-hoc pairwise Mann-Whitney U — PC2 (BH-FDR corrected)")
print(posthoc[['group_1','group_2','n_g1','n_g2','p_raw','q_value','significant']].to_string(index=False))

In [ ]:
# ── Save post-hoc results ──

OUT_POSTHOC = os.path.join(OUT, "tempted_posthoc_pc2.txt")

with open(OUT_POSTHOC, 'w') as f:
    f.write("Post-hoc pairwise Mann-Whitney U — PC2 (BH-FDR corrected)\n\n")
    f.write(posthoc[['group_1','group_2','n_g1','n_g2','p_raw','q_value','significant']].to_string(index=False))

print(f"Saved: {OUT_POSTHOC}")